In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
from pathlib import Path
import copy
import glob 

In [ ]:
path = "../data/raw/"
file = "AMF_CA-Ca2_FLUXNET_2000-2010_v1.3_r1.zip"  # will be substituted by container
keep = ["TIMESTAMP_START", "TIMESTAMP_END", "TA_F", "SW_IN_F", "VPD_F", "P_F",
        "NETRAD", "G_F_MDS", "H_F_MDS", "H_F_MDS_QC", "LE_F_MDS", "LE_F_MDS_QC",
        "NEE_VUT_REF", "NEE_VUT_REF_QC", "GPP_NT_VUT_REF", "RECO_NT_VUT_REF"]

def unpack_data(path="../data/raw/", keep=keep):
    container_list = sorted(glob.glob(path + "*.zip"))
    dflist = []

    for container in container_list:
        with zipfile.ZipFile(container) as archive:
            names = archive.namelist()
            ds_name = next(name for name in names if "FLUXMET_HH" in name)
            bif_name = next(name for name in names if "_BIF_" in name)
            bif = pd.read_csv(archive.open(bif_name))
            igbp_code = bif.loc[bif["VARIABLE"].eq("IGBP"), "DATAVALUE"].iloc[0]

            dataset = pd.read_csv(
                archive.open(ds_name),
                na_values=[-9999],
                usecols=lambda s: s in keep,
                dtype={"TIMESTAMP_START": str, "TIMESTAMP_END": str},
            )

        dataset["time"] = pd.to_datetime(dataset["TIMESTAMP_START"], format="%Y%m%d%H%M")
        dataset["site"] = ds_name.split("_FLUXNET_FLUXMET_HH")[0]
        dataset["dsname"] = ds_name
        dataset["IGBP"] = igbp_code
        dflist.append(dataset)

    return pd.concat(dflist)

master_df = unpack_data()

### C1. Site Overview
A table with one row per tower: site ID, name, country, IGBP class, latitude, longitude, first and last year, hub, and the data citation. Most of it is in your snapshot CSV. Add a simple map of the five sites (a world map with five points is enough).

In [ ]:
# site ID, name, country, IGBP class, latitude, longitude, first and last year, hub, and the data citation
IGBP_CLASSES = {
    "ENF": "Evergreen Needleleaf Forests",
    "EBF": "Evergreen Broadleaf Forests",
    "DNF": "Deciduous Needleleaf Forests",
    "DBF": "Deciduous Broadleaf Forests",
    "MF":  "Mixed Forests",
    "CSH": "Closed Shrublands",
    "OSH": "Open Shrublands",
    "WSA": "Woody Savannas",
    "SAV": "Savannas",
    "GRA": "Grasslands",
    "WET": "Permanent Wetlands",
    "CRO": "Croplands",
    "URB": "Urban and Built-Up Lands",
    "CVM": "Cropland/Natural Vegetation Mosaics",
    "SNO": "Snow and Ice",
    "BSV": "Barren Sparse Vegetation",
    "WAT": "Water Bodies",
                }
countries_dict = {  
                    "CA" : "Canada",
                    "US" : "USA",
                    "FR" : "France",
                    "CN" : "China",
                    "AU" : "Australia"
                }

countries_acronym = master_df["site"].str.split("_").str[-1].str.split("-").str[0]
master_df["country"] = countries_acronym.map(countries_dict)
master_df["IGBP_Class"] = master_df["IGBP"].map(IGBP_CLASSES)

In [ ]:
sites_df = (
    master_df.assign(year=master_df["time"].dt.year)
    .groupby("site", as_index=False)
    .agg(
        country=("country", "first"),
        IGBP=("IGBP", "first"),
        IGBP_Class=("IGBP_Class", "first"),
        first_year=("year", "min"),
        last_year=("year", "max"),
    )
    .sort_values("site")
)

snapshot = pd.read_csv("../data/snapshots/fluxnet_shuttle_snapshot_20261113T110345.csv")
sites_df["site_id"] = sites_df["site"].str.split("_").str[-1]

sites_df = sites_df.merge(
    snapshot[["site_id", "location_lat", "location_long",
              "data_hub", "product_citation"]],
    on="site_id",
    how="left",
).rename(columns={
    "location_lat": "latitude",
    "location_long": "longitude",
    "data_hub": "hub",
    "product_citation": "citation",
})

sites_df

,site,country,IGBP,IGBP_Class,first_year,last_year,site_id,latitude,longitude,hub,citation
0,AMF_CA-Ca2,Canada,ENF,Evergreen Needleleaf Forests,2000,2010,CA-Ca2,49.870500,-125.29090,AmeriFlux,"T. Andrew Black (2025), AmeriFlux FLUXNET-1F C..."
1,AMF_US-Bi1,USA,CRO,Croplands,2016,2024,US-Bi1,38.099200,-121.49930,AmeriFlux,"Camilo Rey-Sanchez, Carlos Tianxin Wang, Daphn..."
2,ICOS_FR-Hes,France,DBF,Deciduous Broadleaf Forests,1997,2026,FR-Hes,48.674100,7.06465,ICOS,"Cuntz, M., Joetzjer, E., Aiguier, T., Courtois..."
3,JPF_CN-HaM,China,GRA,Grasslands,2001,2014,CN-HaM,37.607433,101.33200,ICOS,"Du, M., Gu, S., Kato, T., Li, Y., Liang, Z., Z..."
4,TERN_AU-Dry,Australia,SAV,Savannas,2009,2025,AU-Dry,-15.258800,132.37060,TERN,"Beringer, J., Hutley, L., Northwood, M. (2026)..."


In [32]:
import plotly.express as px

fig = px.scatter_geo(
    sites_df,
    lat="latitude",
    lon="longitude",
    hover_name="site_id",
    projection="natural earth",
)
fig.show()


ModuleNotFoundError: No module named 'plotly'


### C2. Data coverage

For each site and year, the share of half hours in which NEE was measured (`NEE_VUT_REF_QC == 0`). Show it as a heat map: sites on the y-axis, years on the x-axis, colour = share measured. Add one sentence: which site and years have the best data?



### C3. Summary statistics

A table per site with mean, minimum, maximum and % missing for `TA_F`, `SW_IN_F`, `VPD_F`, `P_F`, `NEE_VUT_REF`, `GPP_NT_VUT_REF`, `RECO_NT_VUT_REF`, `H_F_MDS` and `LE_F_MDS`. Give the units in the column headers. For precipitation, give the mean annual sum instead of the mean.



### C4. Time series

Aggregate the half-hourly data to daily values yourself and plot daily NEE, GPP and RECO over the whole record, one panel per site.

Unit conversion: 1 µmol CO₂ m⁻² s⁻¹ = 1.0378 g C m⁻² d⁻¹ (× 86 400 s × 12.011 g mol⁻¹ × 10⁻⁶). So a daily mean in µmol m⁻² s⁻¹ × 1.0378 gives g C m⁻² d⁻¹.

Check your aggregation once against the daily `FLUXMET_DD` file for one site.



### C5. Diurnal cycles

The mean daily course (hour of day on the x-axis) of NEE and LE for each season, all five sites in one figure. Careful: seasons are reversed in the southern hemisphere. Use latitude to define the seasons correctly and say how you did it.



### C6. Your reading

1. **Energy balance** for one tower: scatter of `H_F_MDS + LE_F_MDS` against `NETRAD − G_F_MDS`, using only half hours where H and LE were measured (`_QC == 0`). Add the 1:1 line and the slope of a linear fit. Some towers have no `NETRAD` or `G`; pick one that has them.
2. **5–8 sentences** comparing your towers: which takes up the most carbon per year, which has the largest seasonal cycle, which surprised you, and how much of the data is gap-filled.

Close the report with a **Data and citation** section: each site's citation from the snapshot, Pastorello et al. (2020), *Scientific Data* 7, 225, <https://doi.org/10.1038/s41597-020-0534-3>, and the FLUXNET acknowledgement from `flux4student/README.md`.



### Hand-in and checklist

Submit these two things on ILIAS before Block 5:

1. The rendered HTML report.
2. The link to your fork on GitHub (`https://github.com/YOUR-USER/datascience_env_monit`) with the report's source.

Tick everything before you submit:

- [ ] My fork is synced with the course repository (`git remote -v` shows `upstream`)
- [ ] Five zips in `data/raw`, untouched; `data/` is in `.gitignore`
- [ ] −9999 is read as missing and time stamps are parsed as dates
- [ ] The report names the snapshot file and the file versions (for example `v1.3_r1`)
- [ ] The report says which variables you used (`_VUT_REF`, night-time or day-time partitioning)
- [ ] Every figure has axis labels with units and a site label
- [ ] The six sections C1–C6 are there
- [ ] Data citations and the FLUXNET acknowledgement are in the report
- [ ] It renders from a fresh clone with the commands in your README
